In [ ]:
from openai import api_key
from langchain_openai import ChatOpenAI
from dotenv import dotenv_values

env_values = dotenv_values("./app.env")

llm = ChatOpenAI(
    api_key=env_values[api_key],
    base_url="https://openrouter.ai/api/v1",
    model="openrouter/free",
    temperature=0.8

)

In [39]:
#Loading data:

#CV:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("TestCV.pdf")
cv_docs = loader.load()

#Github (AI Helped in the code to retreive the person whole Github and search every repo details for maximum knowledge):
import requests
from langchain_core.documents import Document

username = "Engineer9336"

repos_url = f"https://api.github.com/users/{username}/repos?per_page=100"
repos = requests.get(repos_url).json()

github_docs = []

for repo in repos:
    repo_name = repo["name"]
    description = repo["description"] or ""
    repo_url = repo["html_url"]

    readme_url = f"https://api.github.com/repos/{username}/{repo_name}/readme"

    response = requests.get(
        readme_url,
        headers={"Accept": "application/vnd.github.raw+json"}
    )

    if response.status_code == 200:
        readme = response.text
    else:
        readme = ""

    content = f"""
Repository: {repo_name}

Description:
{description}

README:
{readme}
"""

    doc = Document(
        page_content=content,
        metadata={
            "source": "GitHub",
            "repo": repo_name,
            "url": repo_url
        }
    )

    github_docs.append(doc)

#Website:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://engineer9336.github.io/portfolio/")
website_docs = loader.load()



#Linkin(Because LinkedIn has strict anti-scraping protections, we will download it in form of pdf from linkdin):

linkedin_loader = PyPDFLoader("Linkedin.pdf")
linkedin_docs = linkedin_loader.load()


#Certifications & Extra: #certificates are already in the pdf downloaded from linkdin
"""
loader = PyPDFLoader("")
Extra1 = loader.load()
loader = PyPDFLoader("")
Extra2 = loader.load()
"""



'\nloader = PyPDFLoader("")\nExtra1 = loader.load()\nloader = PyPDFLoader("")\nExtra2 = loader.load()\n'

In [ ]:
all_docs=cv_docs + github_docs + linkedin_docs + website_docs #add the rest if there any

#(The Data of the student we are using for this demonstration was very unclean and messy so this is for it to clean, This is not necessary)
import re

website_docs = loader.load()

for doc in website_docs:
    doc.page_content = re.sub(
        r'\s+',
        ' ',
        doc.page_content
    ).strip()
    
#Chunking
from langchain_classic.text_splitter import TokenTextSplitter
splitter = TokenTextSplitter(
    chunk_size = 200,
    chunk_overlap = 20 
)

chunks = splitter.split_documents(all_docs)

#Embedding:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs = {'device': 'cpu'}
)


#Vectore Store:
from langchain_classic.vectorstores import FAISS 

vectorDB=FAISS.from_documents(chunks,embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [47]:
from langchain_core.prompts import PromptTemplate

temp = """"
You are assistant, Answer the next Question using provided context,
If you don't know the answer, just say you don't know.
answer should be within 200 words or lower only

## conext:
{context}

## Question : 
{Question}
"""

temp = PromptTemplate.from_template(temp)

query = "What projects has Aryan worked on?"  

similar_docs = vectorDB.similarity_search_with_score(query, k=4)
context = []
for i in similar_docs:
    context.append(i[0].page_content) 


prompt = temp.format(context="\n".join(context), Question =query) 

In [48]:
response = llm.invoke(prompt).content

print(response)

Based on the provided context, Aryan Mishra has worked on the following projects:

1. **Cyber Threat Detection** - Developed during his IBM virtual AI internship
2. **Student Performance Prediction** - Also created as part of his IBM virtual AI internship

These two projects are specifically mentioned in the context as examples of his AI/ML work. The portfolio README describes his personal portfolio website as showcasing various projects, skills, and accomplishments in AI/ML, including AI model implementation (AI_MODEL()), generative AI work (GEN_AI), and other machine learning applications. However, specific project names beyond these two internship projects are not fully detailed in the provided text.

His professional summary notes that he is an aspiring AI/ML developer with projects focused on practical technology solutions using Python, artificial intelligence, and machine learning. He has completed an IBM virtual AI internship where he built these two primary projects, and contin

In [49]:
from IPython.display import display, Markdown

display(Markdown(response))

Based on the provided context, Aryan Mishra has worked on the following projects:

1. **Cyber Threat Detection** - Developed during his IBM virtual AI internship
2. **Student Performance Prediction** - Also created as part of his IBM virtual AI internship

These two projects are specifically mentioned in the context as examples of his AI/ML work. The portfolio README describes his personal portfolio website as showcasing various projects, skills, and accomplishments in AI/ML, including AI model implementation (AI_MODEL()), generative AI work (GEN_AI), and other machine learning applications. However, specific project names beyond these two internship projects are not fully detailed in the provided text.

His professional summary notes that he is an aspiring AI/ML developer with projects focused on practical technology solutions using Python, artificial intelligence, and machine learning. He has completed an IBM virtual AI internship where he built these two primary projects, and continues to pursue growth in areas like deep learning and agentic AI.